# 1D J1J2J3 inference: LorentzGRU (seed 111-555) 

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import sys
sys.path.append('../../1_hypnqs_torch_lorentz/utility_lorentz')
from j1j2j3_train_loop_lorentz import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [3]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test_no_tau(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    #  Load the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    # --- PART C: Check performance AFTER loading ---
    with torch.no_grad():
        test_samples_after = wf.sample_no_tau(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2J3_local_energies(wf, N, J1, J2, J3, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2J3_local_energies(wf, N, J1, J2, J3,Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [4]:
N=100
numsamples = 10000
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)
fname = f'results_LorentzGRU_sc=4.0'

## J2=0.0, J3=0.5

Seeds 222, 333 didn't converge.

In [5]:
J2_ = 0.0
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_00_05=-53.9914

In [6]:
seed=111
set_cpu_deterministic(seed)
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.82419967651367-0.004399999976158142j), var E = 0.8848000168800354
DMRG energy  is -53.9914
Time taken=1.212 hrs


In [5]:
seed=444
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_100_LorentzGRU_70_ns=80_MsTrue_s444_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_00_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 331 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-50.077301025390625-0.004900000058114529j), var E = 6.3927001953125
DMRG energy  is -53.9914
Time taken=1.077 hrs


In [6]:
seed=555
spatial_clamp=4.0
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.0|J3=0.5_100_LorentzGRU_70_ns=80_MsTrue_s555_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_00_05)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-53.73320007324219-0.004800000227987766j), var E = 1.128600001335144
DMRG energy  is -53.9914
Time taken=0.866 hrs


## J2=0.2, J3=0.2

Seed 222 didn't converge.

In [7]:
J2_ = 0.2
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_02=-43.5860

In [9]:
spatial_clamp=4.0
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 432 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.705299377441406-0.0003000000142492354j), var E = 0.4537000060081482
DMRG energy  is -43.586
Time taken=1.6 hrs


In [20]:
spatial_clamp=4.0
seed=333
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 201 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.96369934082031-0.0020000000949949026j), var E = 0.42590001225471497
DMRG energy  is -43.586
Time taken=1.4 hrs


In [22]:
spatial_clamp=4.0
seed=444
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-42.90399932861328+0.0052999998442828655j), var E = 0.3783000111579895
DMRG energy  is -43.586
Time taken=1.388 hrs


In [23]:
spatial_clamp=4.0
seed=555
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.2_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-43.1327018737793-0.006800000090152025j), var E = 0.7028999924659729
DMRG energy  is -43.586
Time taken=1.414 hrs


## J2=0.2, J3=0.5

In [9]:
J2_ = 0.2
J3_ = 0.5
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_02_05=-49.6287

In [12]:
spatial_clamp=4.0
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 370 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-49.153099060058594-0.00019999999494757503j), var E = 0.5314000248908997
DMRG energy  is -49.6287
Time taken=1.579 hrs


In [12]:
spatial_clamp=4.0
seed=222
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 487 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.79199981689453+0.00039999998989515007j), var E = 1.3819999694824219
DMRG energy  is -49.6287
Time taken=1.457 hrs


In [13]:
spatial_clamp=4.0
seed=444
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 338 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-46.47090148925781+0.005100000184029341j), var E = 4.855299949645996
DMRG energy  is -49.6287
Time taken=1.42 hrs


In [14]:
spatial_clamp=4.0
seed=555
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.2|J3=0.5_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_02_05, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 593 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-48.85660171508789-0.004100000020116568j), var E = 0.9093000292778015
DMRG energy  is -49.6287
Time taken=1.417 hrs


## J2=0.5, J3=0.2

Seed 222 didn't converge.

In [14]:
J2_ = 0.5
J3_ = 0.2
J2 = +J2_ * np.ones(syssize)
J3 = +J3_ * np.ones(syssize)
Ee_05_02=-38.5473

In [14]:
spatial_clamp=4.0
seed=111
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05_02)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.74679946899414+0j), var E = 0.3783000111579895
DMRG energy  is -38.5473
Time taken=1.477 hrs


In [16]:
spatial_clamp=4.0
seed=333
set_cpu_deterministic(seed)
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 313 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.402000427246094+0.003100000089034438j), var E = 0.06319999694824219
DMRG energy  is -38.5473
Time taken=1.168 hrs


In [17]:
spatial_clamp=4.0
seed=444
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 388 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-38.06650161743164+0.002099999925121665j), var E = 0.19059999287128448
DMRG energy  is -38.5473
Time taken=1.369 hrs


In [18]:
spatial_clamp=4.0
seed=555
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)

h_wl = f'{fname}/J2={J2_}_J3={J3_}/seed={seed}/N100_J1=1.0|J2=0.5|J3=0.2_100_LorentzGRU_70_ns=80_MsTrue_s{seed}_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_05_02, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 534 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-37.89179992675781+0.00139999995008111j), var E = 0.2881999909877777
DMRG energy  is -38.5473
Time taken=1.394 hrs
